# GreenInfer - Demo Notebook

**GreenInfer** estimates the CO₂ and water footprint of a single AI inference query,
broken down by country (209 countries, Ember Yearly Electricity Data) and all 31 Chinese provinces (NBS data).

- Paper: *GreenInfer + EcoQuery: Estimating the CO₂ and Water Footprint of AI Inference Queries*
- Author: Gayani Upeksha
- Repo: [GanyaGit/GreenInfer](https://github.com/GanyaGit/GreenInfer)

**Key constants (Patterson et al. 2021 + Ren et al. 2025):**
- Energy per query: 0.001 kWh
- Water per query: ~4.91 mL (global median WUE, same for all regions)
- CO₂ varies by grid intensity of the data center location

In [4]:
# Install GreenInfer directly from GitHub
!pip install git+https://github.com/GanyaGit/GreenInfer.git -q

print("Installed successfully.")

Installed successfully.


In [6]:
import greeninfer
from greeninfer.estimator import _load_country_intensities, _load_latest_year_per_country
from greeninfer.regions import CHINA_PROVINCES_BY_YEAR

print("GreenInfer loaded successfully.")
print(f"Province years available: {sorted(CHINA_PROVINCES_BY_YEAR.keys())}")

GreenInfer loaded successfully.
Province years available: [2024, 2025]


## 1. Estimate CO₂ + water by country (Ember data)

Uses each country's latest available year automatically.

In [7]:
countries = ["Germany", "France", "India", "United States", "China", "Sri Lanka", "Japan", "Australia"]

latest_years = _load_latest_year_per_country()

print(f"{'Country':<22} {'Year':<6} {'Intensity (gCO2/kWh)':<24} {'CO2/query (g)':<16} {'Water/query (mL)'}")
print("-" * 85)

for c in countries:
    year = latest_years.get(c, 2023)
    r = greeninfer.estimate(country=c, year=year)
    co2    = r.get('co2_grams',     'N/A')
    water  = r.get('water_ml',      'N/A')
    intens = r.get('co2_intensity', 'N/A')
    print(f"{c:<22} {str(year):<6} {str(intens):<24} {str(co2):<16} {water}")

Country                Year   Intensity (gCO2/kWh)     CO2/query (g)    Water/query (mL)
-------------------------------------------------------------------------------------
Germany                2025   334.1                    0.3341           4.9086
France                 2025   41.2                     0.0412           4.9086
India                  2025   671.0                    0.671            4.9086
United States          2025   384.4                    0.3844           4.9086
China                  2025   528.3                    0.5283           4.9086
Sri Lanka              2024   404.0                    0.404            4.9086
Japan                  2025   473.7                    0.4737           4.9086
Australia              2025   518.0                    0.518            4.9086


## 2. China province-level estimate

Uses NBS monthly electricity generation data, disaggregated by source.
Intensity = (thermal_annual / total_annual) × 820 gCO₂/kWh (IPCC AR5).

In [8]:
provinces = [
    "Inner Mongolia",  # high coal
    "Shanxi",          # coal-dominant
    "Beijing",
    "Shanghai",
    "Guangdong",
    "Sichuan",         # hydro-rich
    "Xizang",          # lowest intensity
]

for year in [2024, 2025]:
    print(f"\n--- Year {year} ---")
    print(f"{'Province':<20} {'Intensity (gCO2/kWh)':<24} {'CO2/query (g)':<16} {'Water/query (mL)'}")
    print("-" * 72)
    for p in provinces:
        r = greeninfer.estimate(country="China", province=p, year=year)
        print(f"{p:<20} {str(r['co2_intensity']):<24} {str(r['co2_grams']):<16} {r['water_ml']}")


--- Year 2024 ---
Province             Intensity (gCO2/kWh)     CO2/query (g)    Water/query (mL)
------------------------------------------------------------------------
Inner Mongolia       624.0                    0.624            4.9086
Shanxi               687.0                    0.687            4.9086
Beijing              795.0                    0.795            4.9086
Shanghai             794.0                    0.794            4.9086
Guangdong            583.0                    0.583            4.9086
Sichuan              147.0                    0.147            4.9086
Xizang               23.0                     0.023            4.9086

--- Year 2025 ---
Province             Intensity (gCO2/kWh)     CO2/query (g)    Water/query (mL)
------------------------------------------------------------------------
Inner Mongolia       579.0                    0.579            4.9086
Shanxi               663.0                    0.663            4.9086
Beijing              787.0

## 3. All 31 provinces - ranked by CO₂ intensity (2025)

In [9]:
year = 2025
prov_data = CHINA_PROVINCES_BY_YEAR[year]

ranked = sorted(prov_data.items(), key=lambda x: -x[1])

print(f"{'Rank':<6} {'Province':<22} {'Intensity (gCO2/kWh)':<24} {'CO2/query (g)'}")
print("-" * 65)
for i, (prov, intensity) in enumerate(ranked, 1):
    co2_g = round(0.001 * intensity, 4)
    print(f"{i:<6} {prov:<22} {intensity:<24} {co2_g}")

Rank   Province               Intensity (gCO2/kWh)     CO2/query (g)
-----------------------------------------------------------------
1      Beijing                787                      0.787
2      Shanghai               784                      0.784
3      Tianjin                729                      0.729
4      Anhui                  719                      0.719
5      Shaanxi                701                      0.701
6      Jiangxi                670                      0.67
7      Shanxi                 663                      0.663
8      Jiangsu                653                      0.653
9      Shandong               651                      0.651
10     Henan                  650                      0.65
11     Zhejiang               611                      0.611
12     Chongqing              610                      0.61
13     Ningxia                593                      0.593
14     Guangdong              589                      0.589
15     Inner M

## 4. Historical comparison - a country across years

In [10]:
country = "China"
years   = [2015, 2017, 2019, 2021, 2023, 2024]

print(f"CO₂ per AI query in {country} over time:")
print(f"{'Year':<8} {'Intensity (gCO2/kWh)':<24} {'CO2/query (g)'}")
print("-" * 45)
for y in years:
    r = greeninfer.estimate(country=country, year=y)
    if r['co2_intensity']:
        print(f"{y:<8} {r['co2_intensity']:<24} {r['co2_grams']}")
    else:
        print(f"{y:<8} No data")

CO₂ per AI query in China over time:
Year     Intensity (gCO2/kWh)     CO2/query (g)
---------------------------------------------
2015     669.7                    0.6697
2017     646.3                    0.6463
2019     622.0                    0.622
2021     602.3                    0.6023
2023     586.4                    0.5864
2024     559.0                    0.559


## 5. Manual factor fallback

For any location not in the dataset, pass a manual CO₂ intensity value.

In [11]:
# Example: custom data center with known grid intensity
r = greeninfer.estimate(country="CustomLocation", manual_factor=550)
print(f"CO2/query:   {r['co2_grams']} g")
print(f"Water/query: {r['water_ml']} mL")
print(f"Method:      {r['method']}")

CO2/query:   0.55 g
Water/query: 4.9086 mL
Method:      manual


## 6. Water footprint - why it's the same everywhere

Water consumption uses global median WUE constants from Ren et al. (2025):
- PUE = 1.4 (Power Usage Effectiveness)
- WUE_onsite = 0.25 L/kWh
- WUE_offsite = 4.73 L/kWh
- Formula: `water = ((E/PUE) × WUE_onsite + E × WUE_offsite) × 1000` mL

In [12]:
from greeninfer.estimator import ENERGY_PER_QUERY_KWH, PUE, WUE_ONSITE, WUE_OFFSITE

E = ENERGY_PER_QUERY_KWH
onsite  = (E / PUE) * WUE_ONSITE  * 1000
offsite =  E        * WUE_OFFSITE  * 1000
total   = round(onsite + offsite, 4)

print(f"Energy per query : {E} kWh")
print(f"On-site water    : {round(onsite, 4)} mL")
print(f"Off-site water   : {round(offsite, 4)} mL")
print(f"Total water      : {total} mL  ← same for every location")

Energy per query : 0.001 kWh
On-site water    : 0.1786 mL
Off-site water   : 4.73 mL
Total water      : 4.9086 mL  ← same for every location
